In [1]:
## 引入必要的库

In [2]:
import os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from typing import Optional,Dict,Any

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported")

Libraries imported


In [3]:
## 配置模型

In [4]:
# 1. 加载同目录下的 .env 文件
load_dotenv()

# 2. 从系统环境变量中读取模型名称（如果没读到，默认用 deepseek-chat）
# 注意：LiteLlm 在底层会自动去寻找 os.environ["DEEPSEEK_API_KEY"]，所以我们甚至不需要手动赋给它！
MODEL_NAME = os.getenv("DEEPSEEK_MODEL", "deepseek/deepseek-chat")
llm = LiteLlm(model=MODEL_NAME)

print(llm.llm_client.completion(model=llm.model,
                                messages=[{"role": "user", "content": "你好，请问你准备好开始了吗？"}],
                                tools=[]))
print("🤖 DeepSeek 回复：")

print("\nDeepSeek is ready for use.")

ModelResponse(id='2f6f9dd6-b2f1-44e3-b2ce-4a45d5c72188', created=1788191289, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='你好！我已经准备好了，随时可以开始。请告诉我你需要什么帮助，无论是回答问题、提供建议，还是协助完成某项任务，我都会尽力协助你！ 😊', role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=34, prompt_tokens=12, total_tokens=46, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=12))
🤖 DeepSeek 回复：

DeepSeek is ready for use.


In [5]:
## 定义用户意图Agent

In [11]:
agent_role_and_goal = """
你是一名知识图谱用例方面的专家。
你的主要目标是帮助用户构思一个知识图谱用例。
"""

In [12]:
agent_conversational_hints = """
如果用户不确定该怎么做，请基于经典场景提供一些建议：
- 包含朋友、家人或职业关系的社交网络
- 包含供应商、客户和合作伙伴的物流网络
- 基于客户、产品和购买模式的推荐系统
- 针对具有可疑交易模式的多个账户的欺诈检测
- 包含电影、书籍或音乐的流行文化图谱
"""

In [13]:
agent_output_definition = """
一个用户目标包含两个组成部分：
- kind_of_graph：最多用 3 个词描述图谱，例如“社交网络”或“美国货运物流”
- description：用几句话描述该图谱的意图，例如“一个用于货物的动态路由和交付系统。”
"""

In [14]:
# 指定智能体应该遵循的步骤
agent_chain_of_thought_directions = """
仔细思考并与用户协作：
1. 理解用户的目标，即一个包含描述的图谱类型 (kind_of_graph)。
2. 根据需要提出澄清问题。
3. 当你认为已经理解了他们的目标时，使用 'set_perceived_user_goal' 工具来记录你的理解。
4. 将你理解的用户目标展示给用户以供确认。
5. 如果用户同意，使用 'approve_perceived_user_goal' 工具来批准该用户目标。这会将该目标保存在状态 (state) 中的 'approved_user_goal' 键下。
"""

In [15]:
complete_agent_instruction = f"""
    {agent_role_and_goal}
    {agent_conversational_hints}
    {agent_output_definition}
    {agent_chain_of_thought_directions}
"""

print(complete_agent_instruction)


    
你是一名知识图谱用例方面的专家。
你的主要目标是帮助用户构思一个知识图谱用例。

    
如果用户不确定该怎么做，请基于经典场景提供一些建议：
- 包含朋友、家人或职业关系的社交网络
- 包含供应商、客户和合作伙伴的物流网络
- 基于客户、产品和购买模式的推荐系统
- 针对具有可疑交易模式的多个账户的欺诈检测
- 包含电影、书籍或音乐的流行文化图谱

    
一个用户目标包含两个组成部分：
- kind_of_graph：最多用 3 个词描述图谱，例如“社交网络”或“美国货运物流”
- description：用几句话描述该图谱的意图，例如“一个用于货物的动态路由和交付系统。”

    
仔细思考并与用户协作：
1. 理解用户的目标，即一个包含描述的图谱类型 (kind_of_graph)。
2. 根据需要提出澄清问题。
3. 当你认为已经理解了他们的目标时，使用 'set_perceived_user_goal' 工具来记录你的理解。
4. 将你理解的用户目标展示给用户以供确认。
5. 如果用户同意，使用 'approve_perceived_user_goal' 工具来批准该用户目标。这会将该目标保存在状态 (state) 中的 'approved_user_goal' 键下。




In [ ]:
# 定义工具

In [16]:
# 工具：设置感知的用户目标 (Set Perceived User Goal)
# 为了鼓励与用户协作，第一个工具仅设置感知的用户目标 (perceived user goal)
from google.adk.tools.tool_context import ToolContext
from neo4j_for_adk import tool_success, tool_error
PERCEIVED_USER_GOAL = "perceived_user_goal"

def set_perceived_user_goal(kind_of_graph: str, graph_description: str, tool_context: ToolContext):
    """
    设置感知的用户目标，包括图谱类型及其描述。

    Args:
        kind_of_graph: 对图谱类型的 2-3 个词的定义，例如“社交网络”
        graph_description: 对图谱的单段描述，总结其意图
    """
    user_goal_data = {"kind_of_graph": kind_of_graph, "graph_description": graph_description}
    tool_context.state[PERCEIVED_USER_GOAL] = user_goal_data
    return tool_success(PERCEIVED_USER_GOAL, user_goal_data)

In [17]:
# 工具：批准感知的用户目标 (Approve the perceived user goal)
# 用户的批准（同意）应该触发对这个工具的调用

APPROVED_USER_GOAL = "approved_user_goal"

def approve_perceived_user_goal(tool_context: ToolContext):
    """
    在获得用户批准后，将感知的用户目标记录为已批准的用户目标。

    只有在用户明确批准了感知的用户目标时，才调用此工具。
    """

    # 信任，但要验证 (Trust, but verify)。
    # 要求在批准之前，必须已经设置了感知目标 (perceived goal)。
    # 注意：工具错误 (tool error) 会帮助 Agent 采取纠正措施。
    if PERCEIVED_USER_GOAL not in tool_context.state:
        # 补全了这里的错误提示指导语：
        return tool_error("perceived_user_goal 未设置。请先设置感知的用户目标，如果您不确定，请提出澄清问题。")

    tool_context.state[APPROVED_USER_GOAL] = tool_context.state[PERCEIVED_USER_GOAL]

    return tool_success(APPROVED_USER_GOAL, tool_context.state[APPROVED_USER_GOAL])

In [20]:
user_intent_agent_tools = [set_perceived_user_goal, approve_perceived_user_goal]

In [21]:
user_intent_agent = Agent(
    name="user_intent_agent_v1",  # 一个唯一的、带有版本号的名称
    model=llm,  # 之前在变量中定义好的大语言模型 (LLM)
    description="帮助用户构思知识图谱用例。",  # 描述该智能体的作用（如果有多Agent协作，其他Agent会读取这个描述）
    instruction=complete_agent_instruction,  # 你之前组合好的完整系统指令 (Prompt)
    tools=user_intent_agent_tools,  # 赋予该智能体的工具列表（包含了存草稿和批准目标的工具）
)

# 打印创建成功的提示信息
print(f"Agent '{user_intent_agent.name}' 已创建。")

Agent 'user_intent_agent_v1' 已创建。


In [22]:
from helper import make_agent_caller

user_intent_caller = await make_agent_caller(user_intent_agent)

# 运行初始对话 (Run the Initial Conversation)

session_start = await user_intent_caller.get_session()
print(f"Session Start: {session_start.state}") # 预期此时的状态是空的 (empty)

# 我们需要一个异步函数来等待(await)每次对话过程
async def run_conversation():
    # 通过描述一个图谱需求来开启对话
    await user_intent_caller.chat("""我想要一个物料清单图谱 (BOM图谱)，它包含从供应商到成品的各个层级，并能支持根本原因分析。""", True) # 发送消息并开启详细输出 (verbose output)

    if PERCEIVED_USER_GOAL not in session_start.state:
        # 大模型(LLM)可能提出了一个澄清问题。在此提供更多细节 (more details)
        await user_intent_caller.chat("""我担心可能存在的制造或供应商问题。""", True)

    # 乐观地假设并直接批准该目标。
    await user_intent_caller.chat("批准那个目标。(Approve that goal.)", True)

await run_conversation()

# 对话结束后，获取最终的会话状态
session_end = await user_intent_caller.get_session()

Libraries imported
Session Start: {}

>>>👤 用户: 我想要一个物料清单图谱 (BOM图谱)，它包含从供应商到成品的各个层级，并能支持根本原因分析。

🤖 Agent 正在思考并执行中...

  [Event] Author: user_intent_agent_v1, Type: Event, Final: True, Content: parts=[Part(
  text="""好的！您想要一个物料清单（Bill of Materials）知识图谱，用于追踪从供应商到成品的各个层级，并支持根本原因分析。这是一个很棒的用例！

让我先整理一下我的理解，然后确认一些细节。

**我的初步理解：**
- **图谱类型：** 物料清单（BOM）图谱
- **描述：** 一个追踪从原材料/供应商到最终成品各个层级的物料清单图谱，并支持根本原因分析（例如，当某批产品出现质量问题时，能够追溯到具体的供应商、原材料批次等）。

在进一步确认之前，我想问几个澄清问题：

1. **行业领域：** 您所在的行业是什么？例如制造业、电子产品、汽车、制药等？这可能影响图谱中需要关注的具体属性。

2. **根本原因分析的重点：** 您更关注以下哪类问题的根因分析？
   - 产品质量问题（例如某批产品不合格，追溯到哪个供应商/原材料）
   - 供应链中断（例如某零件缺货，影响到哪些下游产品）
   - 合规性问题（例如某原材料不符合法规，影响哪些产品）

3. **数据粒度：** 您是否需要追踪到具体的批次（batch/lot）级别，还是只需要到供应商和物料类型级别？

4. **额外关系：** 除了组装/包含关系外，是否需要包含其他关系，如替代物料、供应商资质、成本信息等？

请告诉我这些细节，以便我更准确地理解您的需求！"""
)] role='model'
<<< Agent Response: 好的！您想要一个物料清单（Bill of Materials）知识图谱，用于追踪从供应商到成品的各个层级，并支持根本原因分析。这是一个很棒的用例！

让我先整理一下我的理解，然后确认一些细节。

**我的初步理解：**
- **图谱类型：** 物料清单（BOM）图谱
- **描述：** 一个追踪从原材料/供应商到最终成品各个层级的物料清单图谱